# Physics-Guided Explainability Analysis for TE-Q-Transformer
### Integrated Gradients Feature Attributions & Temperature Kinetics Sensitivity on NASA Battery Benchmark

This notebook provides a self-contained, reproducible explainability pipeline for the **TE-Q-Transformer** model using **Integrated Gradients** (Sundararajan et al., Axiomatic Attribution for Deep Networks, ICML 2017) with 50-step Gauss–Legendre quadrature.

#### Key Analysis Modules:
1. **Reference Checkpoint Verification**: Loads the reference pre-trained TE-Q-Transformer model (`models/checkpoints/nasa_teq_transformer_best.pth`).
2. **Physical Baseline Construction**: Constructs a domain-grounded reference cycle $x'$ (training-mean voltage & current, $25^\circ\mathrm{C}$ ambient temperature, initial time $t=0$).
3. **Gauss-Legendre Integrated Gradients**: Evaluates path-integral gradients across 50 quadrature points in batch-parallel fashion.
4. **Axiomatic Completeness Verification**: Verifies $\sum_{i} \text{Attr}_i(x) = F(x) - F(x')$ within numerical tolerance.
5. **Global & Cell-Wise Channel Importance**: Quantifies relative contributions of Voltage, Current, Temperature, and Time.
6. **Aging Trajectory Sensitivity**: Tracks temperature attribution dynamics across battery cycle life for held-out cells (B0018, B0032, B0053-test).

In [ ]:
# ==============================================================================
# 0. SETUP ENVIRONMENT AND REPOSITORY PATHS
# ==============================================================================
import os
import sys
import time
import math
import pickle
from pathlib import Path

# Resolve repository root across local, Kaggle, and Colab
MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / "TE-Q-Transformer",
    Path("/kaggle/working") / "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation",
    Path("/kaggle/working"),
    Path("/content") / "TE-Q-Transformer",
    Path("/content") / "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation",
    Path("/content"),
]

# Auto-discover any Kaggle input mount points
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("*"):
        CANDIDATES.extend([p, p / "TE-Q-Transformer"])
        for sub in p.glob("*"):
            if sub.is_dir():
                CANDIDATES.append(sub)

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")

# Dynamic data directory resolution (works seamlessly on Kaggle and local)
data_candidates = [
    REPO_ROOT / "datasets" / "NASA" / "processed",
    Path("/kaggle/working/TE-Q-Transformer/datasets/NASA/processed"),
    Path("/kaggle/working/datasets/NASA/processed"),
    Path("datasets/NASA/processed"),
]
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("**/B0018_soh.npy"):
        data_candidates.insert(0, p.parent)

DATA_DIR = next((p.resolve() for p in data_candidates if p.exists() and (p / "B0018_soh.npy").exists()), (REPO_ROOT / "datasets" / "NASA" / "processed").resolve())
print(f"[Setup] DATA_DIR resolved to: {DATA_DIR}")

# Safe writable output directory (avoids Kaggle read-only filesystem issues)
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working") / "outputs" / "explainability"
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content") / "outputs" / "explainability"
else:
    OUTPUT_DIR = REPO_ROOT / "outputs" / "explainability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Setup] OUTPUT_DIR resolved to: {OUTPUT_DIR}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Ensure PennyLane is installed
try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

def seed_everything(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware execution device: {DEVICE}")


In [ ]:
# ==============================================================================
# 1. LOAD MODEL CHECKPOINT AND DATASET SCALER
# ==============================================================================
from models.proposed.te_q_transformer import TEQTransformer, rich_entangler_config
from src.data.nasa_loader import (
    NASA_TRAIN_CELLS,
    load_full_cell,
    split_cell_70_30,
    transform_with_scaler,
)

scaler_path = DATA_DIR / "nasa_train_scaler.pkl"
if not scaler_path.exists():
    scaler_path = DATA_DIR / "nasa_train_scaler_sklearn1.8.0.pkl"

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)
print(f"[Setup] Loaded fitted training scaler from: {scaler_path.name}")

# Checkpoint resolution across local and Kaggle mounts
CKPT_CANDIDATES = [
    REPO_ROOT / "models" / "checkpoints" / "nasa_teq_transformer_best.pth",
    REPO_ROOT / "artifacts" / "nasa" / "01_rich_entangler" / "nasa_teq_transformer_best.pth",
    Path("/kaggle/working/TE-Q-Transformer/models/checkpoints/nasa_teq_transformer_best.pth"),
]
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("**/nasa_teq_transformer_best.pth"):
        CKPT_CANDIDATES.insert(0, p)

ckpt_path = next((p.resolve() for p in CKPT_CANDIDATES if p.exists()), None)
cfg = rich_entangler_config()
model = TEQTransformer(cfg).to(DEVICE)

if ckpt_path:
    print(f"[Setup] Loading reference trained weights from: {ckpt_path}")
    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
else:
    print("[Notice] Checkpoint not found. Running with standard initialized TE-Q-Transformer.")

model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"[Setup] Model initialized and set to evaluation mode on {DEVICE}.")


In [ ]:
# ==============================================================================
# 2. LOAD NASA TEST PARTITIONS
# ==============================================================================
# Test partitions evaluated in the manuscript:
# 1. B0018: 24°C Unseen Cell (132 cycles)
# 2. B0032: 4°C Unseen Cell (39 cycles)
# 3. B0053_test: 44°C Temporal Extrapolation (final 30% of cycle life, 16 cycles)

test_splits = {}

# B0018
X_b18, y_b18 = load_full_cell(DATA_DIR, "B0018")
X_b18_s = transform_with_scaler(X_b18, scaler, clip=True).to(DEVICE)
test_splits["B0018"] = {
    "X": X_b18_s,
    "y": y_b18.to(DEVICE),
    "cell_id": "B0018",
    "temp_c": 24.0,
    "type": "Unseen Cell (24°C)",
}

# B0032
X_b32, y_b32 = load_full_cell(DATA_DIR, "B0032")
X_b32_s = transform_with_scaler(X_b32, scaler, clip=True).to(DEVICE)
test_splits["B0032"] = {
    "X": X_b32_s,
    "y": y_b32.to(DEVICE),
    "cell_id": "B0032",
    "temp_c": 4.0,
    "type": "Unseen Cell (4°C)",
}

# B0053 Temporal Extrapolation
_, (X_b53_test, y_b53_test) = split_cell_70_30(DATA_DIR, "B0053")
X_b53_test_s = transform_with_scaler(X_b53_test, scaler, clip=True).to(DEVICE)
test_splits["B0053_test"] = {
    "X": X_b53_test_s,
    "y": y_b53_test.to(DEVICE),
    "cell_id": "B0053_test",
    "temp_c": 44.0,
    "type": "Temporal Extrapolation (44°C)",
}

total_test_cycles = sum(len(s["y"]) for s in test_splits.values())
print(f"[Data] Total test cycles loaded: {total_test_cycles}")
for k, v in test_splits.items():
    print(f"  - {k} ({v['type']}): {len(v['y'])} cycles, Temp: {v['temp_c']}°C")


In [ ]:
# ==============================================================================
# 3. CONSTRUCT PHYSICAL REFERENCE BASELINE (x')
# ==============================================================================
# Following Sundararajan et al. (2017), the baseline x' represents a physically neutral state:
# - Voltage (Ch 0): Mean discharge voltage across training cells
# - Current (Ch 1): Mean discharge current across training cells
# - Temperature (Ch 2): Reference standard room temperature (25.0 °C)
# - Normalized Time (Ch 3): Initial discharge cycle start point (t = 0.0)

train_X_list = []
for cell_id in NASA_TRAIN_CELLS:
    X_raw, _ = load_full_cell(DATA_DIR, cell_id)
    train_X_list.append(X_raw)
(X_b53_tr, _), _ = split_cell_70_30(DATA_DIR, "B0053")
train_X_list.append(X_b53_tr)
all_train_X = torch.cat(train_X_list, dim=0)
train_X_scaled = transform_with_scaler(all_train_X, scaler, clip=True)

# Compute training means
mean_v = float(train_X_scaled[:, :, 0].mean().item())
mean_i = float(train_X_scaled[:, :, 1].mean().item())

baseline_primary = torch.zeros(1, 512, 4, dtype=torch.float32, device=DEVICE)
baseline_primary[0, :, 0] = mean_v
baseline_primary[0, :, 1] = mean_i
baseline_primary[0, :, 2] = 25.0  # Physical temperature 25.0 °C
baseline_primary[0, :, 3] = 0.0   # Cycle start

with torch.no_grad():
    f_baseline = model(baseline_primary).item()
print(f"[Baseline] Reference Baseline F(x'): {f_baseline:.4f}")


In [ ]:
# ==============================================================================
# 4. INTEGRATED GRADIENTS WITH GAUSS-LEGENDRE QUADRATURE
# ==============================================================================
def compute_integrated_gradients_gauss(model, x, baseline, steps=50):
    """
    Computes Integrated Gradients path attributions using Gauss-Legendre quadrature.
    
    Parameters:
        model: Target PyTorch neural network
        x: Input cycle tensor [1, 512, 4]
        baseline: Reference baseline tensor [1, 512, 4]
        steps: Number of quadrature integration nodes (default: 50)
        
    Returns:
        attributions: Attributions per time-step and channel [1, 512, 4]
    """
    nodes, weights = np.polynomial.legendre.leggauss(steps)
    alphas = torch.tensor(0.5 * (nodes + 1.0), dtype=torch.float32, device=x.device)
    quad_weights = torch.tensor(0.5 * weights, dtype=torch.float32, device=x.device)

    # Parallel batch interpolation across all quadrature nodes
    interp = baseline + alphas.view(-1, 1, 1) * (x - baseline)  # [steps, 512, 4]
    interp = interp.clone().detach().requires_grad_(True)

    preds = model(interp)  # [steps]
    grads = torch.autograd.grad(preds.sum(), interp)[0]  # [steps, 512, 4]

    weighted_grads = (grads * quad_weights.view(-1, 1, 1)).sum(dim=0, keepdim=True)  # [1, 512, 4]
    attributions = (x - baseline) * weighted_grads  # [1, 512, 4]
    return attributions

# Test on a single sample cycle and verify numerical completeness
sample_x = test_splits["B0018"]["X"][0:1]
sample_y = test_splits["B0018"]["y"][0].item()
sample_pred = model(sample_x).item()

t0 = time.time()
attr_sample = compute_integrated_gradients_gauss(model, sample_x, baseline_primary, steps=50)
ig_time = time.time() - t0

sum_attr = attr_sample[0].sum().item()
target_diff = sample_pred - f_baseline
completeness_err = abs(sum_attr - target_diff)

print(f"[Verification] IG computation completed in {ig_time:.3f}s for 50 steps.")
print(f"  - Predicted SOH: {sample_pred:.5f} | Ground Truth: {sample_y:.5f}")
print(f"  - Sum of Attributions: {sum_attr:.5f} | F(x) - F(x'): {target_diff:.5f}")
print(f"  - Completeness Error: {completeness_err:.2e} (Axiom satisfied: {completeness_err < 1e-3})")


In [ ]:
# ==============================================================================
# 5. BATCH ATTRIBUTION EVALUATION ACROSS ALL TEST CYCLES
# ==============================================================================
channels = ["Voltage", "Current", "Temperature", "Time"]
records = []

print("[Analysis] Computing Integrated Gradients across all held-out test cycles...")
t_all = time.time()

for split_id, split in test_splits.items():
    X = split["X"]
    y = split["y"]
    cell_id = split["cell_id"]
    temp_c = split["temp_c"]
    eval_type = split["type"]

    for i in range(len(y)):
        x_i = X[i : i + 1]
        true_soh = y[i].item()
        with torch.no_grad():
            pred_soh = model(x_i).item()

        attr = compute_integrated_gradients_gauss(model, x_i, baseline_primary, steps=50)
        abs_attr = attr[0].abs().sum(dim=0).cpu().numpy()  # Sum over 512 temporal points for each of 4 channels
        signed_attr = attr[0].sum(dim=0).cpu().numpy()
        total_abs = abs_attr.sum()
        pct_attr = (abs_attr / (total_abs + 1e-12)) * 100.0

        records.append({
            "cell_id": cell_id,
            "cycle_index": i + 1,
            "temperature_c": temp_c,
            "eval_type": eval_type,
            "true_soh": true_soh,
            "pred_soh": pred_soh,
            "error": abs(pred_soh - true_soh),
            "attr_voltage_abs": abs_attr[0],
            "attr_current_abs": abs_attr[1],
            "attr_temp_abs": abs_attr[2],
            "attr_time_abs": abs_attr[3],
            "pct_voltage": pct_attr[0],
            "pct_current": pct_attr[1],
            "pct_temp": pct_attr[2],
            "pct_time": pct_attr[3],
            "signed_temp_attr": signed_attr[2],
        })

df_ig = pd.DataFrame(records)
print(f"[Analysis] Processed {len(df_ig)} cycles in {time.time()-t_all:.2f}s.")
csv_out = OUTPUT_DIR / "integrated_gradients_test_attributions.csv"
df_ig.to_csv(csv_out, index=False)
print(f"[Analysis] Saved cycle attribution results to: {csv_out}")


In [ ]:
# ==============================================================================
# 6. GLOBAL & CELL-WISE FEATURE IMPORTANCE SUMMARY
# ==============================================================================
macro_importance = df_ig.groupby("cell_id")[["pct_voltage", "pct_current", "pct_temp", "pct_time"]].mean()
overall_macro = macro_importance.mean(axis=0)

print("=" * 65)
print("CHANNEL IMPORTANCE BREAKDOWN (EQUAL-CELL MACRO AVERAGE):")
print("=" * 65)
for ch, val in zip(channels, overall_macro):
    print(f"  - {ch:<14s}: {val:6.2f}%")
print("=" * 65)

print("\nCELL-SPECIFIC CHANNEL IMPORTANCE:")
print(macro_importance.round(2).to_string())


In [ ]:
# ==============================================================================
# 7. PUBLICATION-QUALITY VISUALIZATION
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=150)

# Panel A: Macro Channel Importance Bar Chart
ax1 = axes[0]
bar_colors = ["#2b5c8f", "#4682b4", "#d95f02", "#7570b3"]
bars = ax1.bar(channels, overall_macro.values, color=bar_colors, width=0.55, edgecolor="black", linewidth=1.1)
for bar in bars:
    h = bar.get_height()
    ax1.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.set_title("(a) Global Feature Importance (Integrated Gradients)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Macro Relative Importance (%)", fontsize=11)
ax1.set_ylim(0, max(overall_macro.values) * 1.2)
ax1.grid(axis="y", linestyle=":", alpha=0.6)

# Panel B: Temperature Attribution Dynamics Across Degradation
ax2 = axes[1]
cell_styles = {"B0018": ("#1f77b4", "B0018 (24°C)"), "B0032": ("#2ca02c", "B0032 (4°C)"), "B0053_test": ("#d62728", "B0053 (44°C)")}
for cid, (col, label) in cell_styles.items():
    sub = df_ig[df_ig["cell_id"] == cid]
    ax2.plot(sub["cycle_index"], sub["pct_temp"], label=label, color=col, linewidth=1.8, alpha=0.85)
ax2.set_title("(b) Temperature Attribution Across Cycle Life", fontsize=12, fontweight="bold")
ax2.set_xlabel("Cycle Index", fontsize=11)
ax2.set_ylabel("Temperature Attribution (%)", fontsize=11)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(frameon=True, facecolor="white", framealpha=0.9)

plt.tight_layout()
fig_png = OUTPUT_DIR / "fig_explainability_analysis.png"
fig_pdf = OUTPUT_DIR / "fig_explainability_analysis.pdf"
plt.savefig(fig_png, bbox_inches="tight")
plt.savefig(fig_pdf, bbox_inches="tight")
plt.show()
print(f"[Figure] Saved explainability figures to:\n  - {fig_png}\n  - {fig_pdf}")


In [ ]:
# ==============================================================================
# ARTIFACT PACKAGING FOR 1-CLICK KAGGLE DOWNLOAD
# ==============================================================================
import shutil
target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
zip_base = str(target_dir / "Explainability_Results")
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"[SUCCESS] Saved 1-click download bundle to: {zip_base}.zip")
print("You can now download Explainability_Results.zip directly from the Kaggle Output viewer!")
